In [2]:
from dash import Dash, dcc, html, Input, Output
import plotly.express as px
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
# import plotly.io as pio
# pio.renderers.default = "notebook_connected"

In [ ]:
S0_8_R0_child = pd.read_csv("output/S0.8-R0__child_results.csv")
S0_8_R0_node = pd.read_csv("output/S0.8-R0__node_stats.csv")
S0_8_R0_parent = pd.read_csv("output/S0.8-R0__parent_results.csv")
S0_8_R0_static = pd.read_csv("output/S0.8-R0__static_info.csv")
S0_8_R0_time = pd.read_csv("output/S0.8-R0__time.csv")
S0_8_R0_unary = pd.read_csv("output/S0.8-R0__unary_results.csv")


S0_8_R1_child = pd.read_csv("output/S0.8-R1__child_results.csv")
S0_8_R1_node = pd.read_csv("output/S0.8-R1__node_stats.csv")
S0_8_R1_parent = pd.read_csv("output/S0.8-R1__parent_results.csv")
S0_8_R1_static = pd.read_csv("output/S0.8-R1__static_info.csv")
S0_8_R1_time = pd.read_csv("output/S0.8-R1__time.csv")
S0_8_R1_unary = pd.read_csv("output/S0.8-R1__unary_results.csv")


listed = {0.2, 0.5, 1.1, 1.4, 1.7, 2.0} # excluding 0.8 bc i didnt realize i only had two of them not four like the rest 

suffixes = ['child_results', 'node_stats', 'parent_results', 'static_info', 'unary_results']

data = {}

for i in listed:
    data[i] = {} 
    for j in range(4):
        data[i][j] = {}
        
        for suffix in suffixes:
        
            file_path = f"output/S{i}-R{j}__{suffix}.csv"
            

  
            data[i][j][suffix] = pd.read_csv(file_path)
          

# Example: How to access the 'node_stats' for S0.2, R0
# df = data[0.2][0]['node_stats']
          

# Example: How to access the 'node_stats' for S0.2, R0
# df = data[0.2][0]['node_stats']

S0_8_unary = pd.concat([S0_8_R0_unary, S0_8_R1_unary], ignore_index=True)
S0_8_child = pd.concat([S0_8_R0_child, S0_8_R1_child], ignore_index=True)
S0_8_node = pd.concat([S0_8_R0_node, S0_8_R1_node], ignore_index=True)
S0_8_parent = pd.concat([S0_8_R0_parent, S0_8_R1_parent], ignore_index=True)

S0_2_unary = pd.concat([data[0.2][j]['unary_results'] for j in range(4)], ignore_index=True)
S0_5_unary = pd.concat([data[0.5][j]['unary_results'] for j in range(4)], ignore_index=True)
S1_1_unary = pd.concat([data[1.1][j]['unary_results'] for j in range(4)], ignore_index=True)
S1_4_unary = pd.concat([data[1.4][j]['unary_results'] for j in range(4)], ignore_index=True)
S1_7_unary = pd.concat([data[1.7][j]['unary_results'] for j in range(4)], ignore_index=True)
S2_0_unary = pd.concat([data[2.0][j]['unary_results'] for j in range(4)], ignore_index=True)


In [6]:
error_change_proportion_by_time_bin(S0_2_unary, data[0.2][0]['static_info'])


In [7]:
error_change_proportion_by_time_bin(S0_5_unary, data[0.5][0]['static_info'])


In [8]:
error_change_proportion_by_time_bin(S0_8_unary, S0_8_R0_static)


In [9]:
error_change_proportion_by_time_bin(S1_1_unary, data[1.1][0]['static_info'])


In [10]:
error_change_proportion_by_time_bin(S1_4_unary, data[1.4][0]['static_info'])


In [11]:
error_change_proportion_by_time_bin(S1_7_unary, data[1.7][0]['static_info'])


In [12]:
error_change_proportion_by_time_bin(S2_0_unary, data[2.0][0]['static_info'])


In [5]:
def error_change_proportion_by_time_bin(results, static):
    sigma = static['sigma'].values[0]
    num_bins = 8

    #correct_span = results.iloc[:, 7] - results.iloc[:, 8]

    
    results['correct_span_bin'], bin_edges = pd.qcut(
        results.iloc[:, 1], 
        q=num_bins, 
        retbins=True, 
        duplicates='drop'
    )

    labels = [f"{int(bin_edges[i])}-{int(bin_edges[i+1])}" for i in range(len(bin_edges)-1)]
    unique_bins = sorted(results['correct_span_bin'].unique())

    fig = make_subplots(rows=2, cols=4, subplot_titles=labels)
    
    for idx, bin_interval in enumerate(unique_bins):
        row = idx // 4 + 1
        col = idx % 4 + 1
    
        bin_data = results[results['correct_span_bin'] == bin_interval]
        proportion = ((bin_data.iloc[:, 2] - bin_data.iloc[:, 3]) / bin_data.iloc[:, 2]) * 100
        
      # where 2 is simp and  3 is extended , 


        fig.add_trace(go.Histogram(
            x=proportion, 
            name='Percent Decrease',
            opacity=0.6, 
            marker_color='red',
            showlegend=(idx == 0)
        ), row=row, col=col)

    fig.update_layout(barmode='overlay', title=f"Sigma {sigma} Error Percent Decrease by Time Bin", height=600)
    fig.update_yaxes(type='log')
    fig.show()
    